In [1]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension --sys-prefix
!jupyter nbextension install --py widgetsnbextension --sys-prefix
!jupyter nbextension enable widgetsnbextension --user --py

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook qtconsole run server
troubleshoot trust

Jupyter command `jupyter-nbextension` not found.
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
         

In [11]:
import pandas as pd

# Load CSV
df = pd.read_csv("AI_Resume_Screening.csv")

In [13]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Rename columns for ease
df.rename(columns={
    'Experience (Years)': 'Experience',
    'Projects Count': 'Projects',
    'Salary Expectation ($)': 'Salary',
    'Recruiter Decision': 'Decision'
}, inplace=True)

# Binary encode 'Decision'
df['Decision'] = df['Decision'].apply(lambda x: 1 if x.lower() == 'hire' else 0)

# Label encode categorical fields
df['Education_Encoded'] = LabelEncoder().fit_transform(df['Education'])
df['JobRole_Encoded'] = LabelEncoder().fit_transform(df['Job Role'])

# Normalize numeric features
scaler = MinMaxScaler()
df[['Experience_Norm', 'Projects_Norm']] = scaler.fit_transform(df[['Experience', 'Projects']])

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
df['Skills'] = df['Skills'].fillna("")
tfidf_matrix = vectorizer.fit_transform(df['Skills'])

import pandas as pd
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f"Skill_{w}" for w in vectorizer.get_feature_names_out()])

In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Predefined job descriptions
job_descriptions = {
    "Data Scientist": """
    Looking for a data scientist with experience in machine learning, Python, data analysis, 
    feature engineering, and cloud computing (AWS or Azure).
    """,
    "Frontend Developer": """
    We are looking for a frontend developer skilled in HTML, CSS, JavaScript, and frameworks like React or Angular. 
    Experience in UI/UX design is a plus.
    """,
    "Backend Developer": """
    Seeking a backend developer with experience in Node.js, Django, REST APIs, and SQL or NoSQL databases.
    Knowledge of Docker and cloud deployment is a plus.
    """,
    "DevOps Engineer": """
    Searching for a DevOps engineer familiar with CI/CD pipelines, infrastructure as code, monitoring tools, 
    and cloud services like AWS, GCP, or Azure.
    """
}

# Dropdown widget
dropdown = widgets.Dropdown(
    options=job_descriptions.keys(),
    value='Data Scientist',
    description='Job Role:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

# Button to trigger ranking
button = widgets.Button(
    description="Get Top Matches",
    button_style='primary', 
    layout=widgets.Layout(width='30%', margin='10px 0px')
)

# Output area
output = widgets.Output()

# Event handler for button click
def on_button_clicked(b):
    with output:
        clear_output()
        selected_description = job_descriptions[dropdown.value]
        job_vec = vectorizer.transform([selected_description])
        similarity_scores = cosine_similarity(job_vec, tfidf_matrix).flatten()
        df_final['MatchScore'] = similarity_scores
        
        top_matches = df_final.sort_values(by='MatchScore', ascending=False)[
            ['Name', 'Job Role', 'Skills', 'Experience', 'Education', 'MatchScore']
        ].head(10)
        
        file_name = f"Top_Matching_Resumes_{dropdown.value.replace(' ', '')}.csv"
        top_matches.to_csv(file_name, index=False)
        
        print(f"✅ Top matches exported to '{file_name}'\n")
        
        for index, row in top_matches.iterrows():
            candidate_info = widgets.HTML(
                value=f"""
                <b>Name:</b> {row['Name']}<br>
                <b>Job Role:</b> {row['Job Role']}<br>
                <b>Skills:</b> {row['Skills']}<br>
                <b>Experience:</b> {row['Experience']}<br>
                <b>Education:</b> {row['Education']}<br>
                """
            )

            score_percent = row['MatchScore'] * 100
            if score_percent >= 80:
                color = 'success'
            elif score_percent >= 60:
                color = 'warning'
            else:
                color = 'danger'

            progress = widgets.FloatProgress(
                value=row['MatchScore'],
                min=0,
                max=1,
                bar_style=color,
                orientation='horizontal',
                layout=widgets.Layout(width='90%')
            )

            card = widgets.VBox([candidate_info, progress],
                                layout=widgets.Layout(
                                    border='1px solid #ccc',
                                    padding='10px',
                                    margin='10px 0px',
                                    background_color='#f9f9f9',
                                    width='90%'
                                ))
            display(card)

# Link button to function
button.on_click(on_button_clicked)

# Display the dropdown, button, and output area
display(widgets.VBox([dropdown, button, output]))

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df_final = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
features = ['Education_Encoded', 'JobRole_Encoded', 'Experience_Norm', 'Projects_Norm'] + list(tfidf_df.columns)

X = df_final[features]
y = df_final['Decision']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.87      0.89        46
           1       0.96      0.97      0.97       154

    accuracy                           0.95       200
   macro avg       0.94      0.92      0.93       200
weighted avg       0.95      0.95      0.95       200



In [23]:
from sklearn.metrics.pairwise import cosine_similarity

job_description = """
Looking for a data scientist with experience in machine learning, Python, data analysis, 
feature engineering, and cloud computing (AWS or Azure).
"""

job_vec = vectorizer.transform([job_description])
similarity_scores = cosine_similarity(job_vec, tfidf_matrix).flatten()
df_final['MatchScore'] = similarity_scores

top_matches = df_final.sort_values(by='MatchScore', ascending=False)[['Name', 'Job Role', 'Skills', 'MatchScore']].head(10)
print(top_matches)

# Select top matches with additional fields
top_matches = df_final.sort_values(by='MatchScore', ascending=False)[
    ['Name', 'Job Role', 'Skills', 'Experience', 'Education', 'MatchScore']
].head(10)

# Export to CSV
top_matches.to_csv("Top_Matching_Resumes.csv", index=False)
print("Top matches exported to 'Top_Matching_Resumes.csv'")

                      Name        Job Role                    Skills  \
475        Christina Oneal  Data Scientist  Python, Machine Learning   
722         Jennifer Moran  Data Scientist  Python, Machine Learning   
531       Eddie Hutchinson  Data Scientist  Python, Machine Learning   
47   Christopher Stevenson  Data Scientist  Python, Machine Learning   
507              Eric Bush  Data Scientist  Machine Learning, Python   
575          James Johnson  Data Scientist  Python, Machine Learning   
930           Mark Sanford  Data Scientist  Python, Machine Learning   
596         Samantha Smith  Data Scientist  Python, Machine Learning   
648            Emily Young  Data Scientist  Machine Learning, Python   
348         Ashley Gardner  Data Scientist  Machine Learning, Python   

     MatchScore  
475         1.0  
722         1.0  
531         1.0  
47          1.0  
507         1.0  
575         1.0  
930         1.0  
596         1.0  
648         1.0  
348         1.0  
Top match